# CSE_CIC_IDS2018_PyTorch_CADE\n\nSplit Kaggle notebook generated from `03_CSE_CIC_IDS2018_PyTorch.ipynb`. This notebook runs only the CADE-style MLP model.\n

## 2. Install/import dependencies

Colab already includes the core scientific Python stack. The install cell is intentionally minimal and can be extended if your runtime is missing a package.

In [ ]:
# Colab usually has these installed. Uncomment if your runtime is missing packages.
# !pip install -q pandas numpy scikit-learn torch

In [ ]:
import os
import json
import math
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

In [ ]:
DATASET_NAME = "CSE-CIC-IDS-2018"
FRAMEWORK = "PyTorch"
NOTEBOOK_FILENAME = "CSE_CIC_IDS2018_PyTorch_CADE.ipynb"
FRAMEWORK_STYLE_MODEL_NAME = "CADE-style MLP"
IMPROVED_MODEL_NAME = "Improved CSE-CIC-IDS-2018 MLP"


## 3. Set reproducibility configuration

The training seeds and weight initialization tuples follow the reproducibility setup from “Randomness Unmasked: Towards Reproducible and Fair Evaluation of Shift-Aware Deep Learning NIDS”.

In [ ]:
TRAINING_SEEDS = [57, 305, 5, 9667, 405]
WEIGHT_INIT_SEEDS = {
    "W1": [1004, 77, 259, 35],
    "W2": [8, 358, 200, 35],
    "W3": [487, 22, 900, 7],
}

DEFAULT_SEED = TRAINING_SEEDS[0]

def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

set_global_seed(DEFAULT_SEED)

## 4. Create data and Kaggle results folders\n\nGenerated CSV results are written to `/kaggle/working/results/`.\n

In [ ]:
DATA_DIR = Path("/content/data")
RESULTS_DIR = Path("/kaggle/working/results")
DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Data directory:", DATA_DIR)
print("Results directory:", RESULTS_DIR)

In [ ]:
RESULTS_CSV_PATH = RESULTS_DIR / "CSE_CIC_IDS2018_PyTorch_CADE_results.csv"
AGGREGATED_RESULTS_CSV_PATH = RESULTS_DIR / "CSE_CIC_IDS2018_PyTorch_CADE_aggregated_results.csv"

print("Per-run results CSV:", RESULTS_CSV_PATH)
print("Aggregated results CSV:", AGGREGATED_RESULTS_CSV_PATH)


## 5. Dataset download with kagglehub

This notebook downloads the Kaggle dataset with `kagglehub`, copies the downloaded files into `/content/data`, preserves subdirectories, and prints discovered CSV files.

In [ ]:
# Colab shell equivalent: !pip install -q kagglehub
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])

In [ ]:
import shutil
import kagglehub

path = kagglehub.dataset_download("solarmainframe/ids-intrusion-csv")
print("Path to dataset files:", path)

source_path = Path(path)
DATA_DIR.mkdir(parents=True, exist_ok=True)

if source_path.is_file():
    target_path = DATA_DIR / source_path.name
    if source_path.resolve() != target_path.resolve():
        shutil.copy2(source_path, target_path)
else:
    for source_item in source_path.rglob("*"):
        if source_item.is_file():
            relative_path = source_item.relative_to(source_path)
            target_path = DATA_DIR / relative_path
            target_path.parent.mkdir(parents=True, exist_ok=True)
            if source_item.resolve() != target_path.resolve():
                shutil.copy2(source_item, target_path)

csv_files = sorted(DATA_DIR.rglob("*.csv"))
print(f"CSV files discovered after copying: {len(csv_files)}")
for csv_file in csv_files:
    print(csv_file)

## 6. Load dataset from /content/data

This loader recursively finds CSV files with `Path("/content/data").rglob("*.csv")`, adds a `source_file` column before concatenation, prints the discovered files, reports dataset shape, and prints label distribution.

In [ ]:
LABEL_CANDIDATES = ["Label", "label", "Class", "class", "Attack", "attack"]

def detect_label_column(df: pd.DataFrame) -> str:
    for candidate in LABEL_CANDIDATES:
        if candidate in df.columns:
            return candidate

    lower_map = {col.lower(): col for col in df.columns}
    for candidate in LABEL_CANDIDATES:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]

    object_columns = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    object_columns = [col for col in object_columns if col != "source_file"]
    if object_columns:
        return object_columns[-1]

    raise ValueError(
        "Could not automatically detect a label column. Rename the target column to one of: "
        + ", ".join(LABEL_CANDIDATES)
    )

def load_csv_dataset(data_dir: Path) -> pd.DataFrame:
    csv_files = sorted(data_dir.rglob("*.csv"))
    print(f"CSV files found: {len(csv_files)}")

    if not csv_files:
        raise FileNotFoundError(
            "No CSV files found under /content/data. Check that kagglehub downloaded successfully "
            "and that files were copied into /content/data."
        )

    frames = []
    for csv_path in csv_files:
        print(f"Loading {csv_path}")
        frame = pd.read_csv(csv_path, low_memory=False)
        frame.columns = frame.columns.astype(str).str.strip()
        frame["source_file"] = str(csv_path.relative_to(data_dir))
        frames.append(frame)

    df = pd.concat(frames, ignore_index=True) if len(frames) > 1 else frames[0]
    df.columns = df.columns.astype(str).str.strip()

    print(f"Dataset shape: {df.shape}")
    print("Column names:")
    print(list(df.columns))

    label_col = detect_label_column(df)
    print(f"Detected label column: {label_col}")
    print("Label distribution:")
    print(df[label_col].astype(str).str.strip().value_counts(dropna=False))
    return df

raw_df = load_csv_dataset(DATA_DIR)
raw_df.head()

## 7. Data preprocessing

Preprocessing is fit after the paper-style train/test protocol is built. The label encoder is fit on training labels when possible, and `StandardScaler` is fit only on `X_train` before transforming `X_test`.

In [ ]:
def clean_label_series(series: pd.Series) -> pd.Series:
    return series.astype(str).str.strip()

def is_benign_label(label: str) -> bool:
    normalized = str(label).strip().lower()
    return normalized in {"benign", "normal", "normal traffic", "background"} or "benign" in normalized

def to_binary_attack_labels(labels: pd.Series) -> pd.Series:
    return clean_label_series(labels).apply(lambda value: "Benign" if is_benign_label(value) else "Attack")

def prepare_feature_matrix(train_df: pd.DataFrame, test_df: pd.DataFrame, label_col: str):
    X_train_df = train_df.drop(columns=[label_col], errors="ignore").copy()
    X_test_df = test_df.drop(columns=[label_col], errors="ignore").copy()

    for frame in (X_train_df, X_test_df):
        frame.drop(columns=["source_file"], inplace=True, errors="ignore")
        for col in frame.columns:
            if frame[col].dtype == "object":
                frame[col] = pd.to_numeric(frame[col].astype(str).str.strip(), errors="coerce")

    numeric_cols = X_train_df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_cols:
        raise ValueError("No numeric feature columns were found after protocol splitting.")

    X_train_df = X_train_df[numeric_cols].replace([np.inf, -np.inf], np.nan)
    X_test_df = X_test_df.reindex(columns=numeric_cols).replace([np.inf, -np.inf], np.nan)

    train_medians = X_train_df.median(numeric_only=True).fillna(0)
    X_train_df = X_train_df.fillna(train_medians).fillna(0)
    X_test_df = X_test_df.fillna(train_medians).fillna(0)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_df).astype(np.float32)
    X_test = scaler.transform(X_test_df).astype(np.float32)
    return X_train, X_test, scaler, numeric_cols

def fit_label_encoder_train_when_possible(y_train_raw: pd.Series, y_test_raw: pd.Series):
    y_train_raw = clean_label_series(y_train_raw)
    y_test_raw = clean_label_series(y_test_raw)
    train_classes = set(y_train_raw.unique())
    test_classes = set(y_test_raw.unique())
    unseen_test_classes = sorted(test_classes - train_classes)

    label_encoder = LabelEncoder()
    if unseen_test_classes:
        print("Test labels not present in training labels; fitting LabelEncoder on train+test labels for transform compatibility:")
        print(unseen_test_classes)
        label_encoder.fit(pd.concat([y_train_raw, y_test_raw], ignore_index=True))
    else:
        label_encoder.fit(y_train_raw)

    return label_encoder, label_encoder.transform(y_train_raw), label_encoder.transform(y_test_raw)

def prepare_protocol_data(train_df: pd.DataFrame, test_df: pd.DataFrame, *, binary_labels: bool = False, metadata: dict | None = None):
    global X_train, X_test, y_train, y_test, label_encoder, scaler, feature_columns
    global NUM_FEATURES, NUM_CLASSES, IS_BINARY, RUN_METADATA

    label_col = detect_label_column(pd.concat([train_df.head(1), test_df.head(1)], ignore_index=True))
    train_df = train_df.copy().replace([np.inf, -np.inf], np.nan).dropna(subset=[label_col])
    test_df = test_df.copy().replace([np.inf, -np.inf], np.nan).dropna(subset=[label_col])

    if binary_labels:
        y_train_raw = to_binary_attack_labels(train_df[label_col])
        y_test_raw = to_binary_attack_labels(test_df[label_col])
    else:
        y_train_raw = clean_label_series(train_df[label_col])
        y_test_raw = clean_label_series(test_df[label_col])

    X_train, X_test, scaler, feature_columns = prepare_feature_matrix(train_df, test_df, label_col)
    label_encoder, y_train, y_test = fit_label_encoder_train_when_possible(y_train_raw, y_test_raw)
    y_train = y_train.astype(np.int64)
    y_test = y_test.astype(np.int64)

    NUM_FEATURES = X_train.shape[1]
    NUM_CLASSES = len(label_encoder.classes_)
    IS_BINARY = NUM_CLASSES == 2
    RUN_METADATA = metadata.copy() if metadata else {}

    print("Train shape:", X_train.shape, y_train.shape)
    print("Test shape:", X_test.shape, y_test.shape)
    print(f"Classes ({NUM_CLASSES}): {list(label_encoder.classes_)}")
    print("Scaler fitted on training data only.")
    return X_train, X_test, y_train, y_test

## 8. Train/test split

CSE-CIC-IDS-2018 uses CADE-style one-class-held-out protocols for every model. Each held-out attack is excluded from training and evaluated against benign/normal test samples.

In [ ]:
HELD_OUT_ATTACKS = ["SSH-Patator", "DoS Hulk", "Infiltration"]

def build_cse_cic_2018_one_class_held_out(df: pd.DataFrame, held_out_attack: str):
    label_col = detect_label_column(df)
    labels = clean_label_series(df[label_col])
    normalized_labels = labels.str.lower()
    held_out_normalized = held_out_attack.strip().lower()
    benign_mask = labels.apply(is_benign_label)
    held_out_mask = normalized_labels == held_out_normalized

    if not held_out_mask.any():
        available = sorted(labels.unique())
        raise ValueError(
            f"Held-out attack '{held_out_attack}' was not found in the label column '{label_col}'. "
            f"Available labels: {available}"
        )

    train_df = df[~held_out_mask].copy()
    test_df = df[benign_mask | held_out_mask].copy()

    if train_df.empty or test_df.empty:
        raise ValueError(f"Invalid split for held-out attack '{held_out_attack}': train={train_df.shape}, test={test_df.shape}")

    print(f"CSE-CIC-IDS-2018 one-class-held-out split: {held_out_attack}")
    print("Training excludes held-out attack class.")
    print("Train label distribution:")
    print(train_df[label_col].astype(str).str.strip().value_counts(dropna=False))
    print("Test label distribution:")
    print(test_df[label_col].astype(str).str.strip().value_counts(dropna=False))
    return train_df, test_df

def prepare_cse_held_out_experiment(held_out_attack: str):
    train_df, test_df = build_cse_cic_2018_one_class_held_out(raw_df, held_out_attack)
    return prepare_protocol_data(
        train_df,
        test_df,
        binary_labels=True,
        metadata={
            "held_out_attack": held_out_attack,
            "train_files": None,
            "test_files": None,
        },
    )

print("Held-out attacks to evaluate:")
for attack in HELD_OUT_ATTACKS:
    print(" -", attack)

## 9. Define evaluation metrics

Metrics use scikit-learn. Multi-class precision, recall, and F1 use `average="weighted"`; binary experiments use binary averaging.

In [ ]:
def metric_average(num_classes: int) -> str:
    return "binary" if num_classes == 2 else "weighted"

def evaluate_predictions(y_true, y_pred, labels=None) -> dict:
    average = metric_average(NUM_CLASSES)
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average=average, zero_division=0),
        "recall": recall_score(y_true, y_pred, average=average, zero_division=0),
        "f1": f1_score(y_true, y_pred, average=average, zero_division=0),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
    }
    try:
        metrics["classification_report"] = classification_report(
            y_true,
            y_pred,
            target_names=labels,
            zero_division=0,
            output_dict=True,
        )
    except Exception:
        metrics["classification_report"] = classification_report(
            y_true,
            y_pred,
            zero_division=0,
            output_dict=True,
        )
    return metrics

def _aggregate_with_group_cols(per_run_df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    metric_cols = ["accuracy", "precision", "recall", "f1"]
    aggregated = per_run_df.groupby(group_cols, dropna=False)[metric_cols].agg(["mean", "min", "max", "std"]).round(6)
    aggregated.columns = [f"{metric}_{stat}" for metric, stat in aggregated.columns]
    return aggregated.reset_index()

def aggregate_results(per_run_df: pd.DataFrame) -> pd.DataFrame:
    base_cols = ["dataset_name", "framework", "model_name"]
    if "held_out_attack" in per_run_df.columns and per_run_df["held_out_attack"].notna().any():
        per_attack = _aggregate_with_group_cols(per_run_df, base_cols + ["held_out_attack"])
        overall = _aggregate_with_group_cols(per_run_df, base_cols)
        overall.insert(len(base_cols), "held_out_attack", "OVERALL")
        return pd.concat([per_attack, overall], ignore_index=True)
    return _aggregate_with_group_cols(per_run_df, base_cols)

## 10. Define weight initialization utilities

Every model is trained with each of the three named weight initialization tuples. These utilities also define the common training loop used by all model sections.

In [ ]:
def make_torch_generator(seed: int) -> torch.Generator:
    generator = torch.Generator()
    generator.manual_seed(seed)
    return generator

def get_dataloaders(X_train, y_train, batch_size: int, seed: int):
    X_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_dtype = torch.float32 if IS_BINARY else torch.long
    y_tensor = torch.tensor(y_train, dtype=y_dtype)
    if IS_BINARY:
        y_tensor = y_tensor.view(-1, 1)
    dataset = TensorDataset(X_tensor, y_tensor)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, generator=make_torch_generator(seed))
    return loader

def apply_weight_initialization(model: nn.Module, seed_tuple) -> None:
    layers_to_init = [m for m in model.modules() if isinstance(m, (nn.Linear, nn.Conv1d))]
    for index, layer in enumerate(layers_to_init):
        torch.manual_seed(seed_tuple[index % len(seed_tuple)])
        if isinstance(layer, nn.Linear):
            nn.init.xavier_uniform_(layer.weight)
        else:
            nn.init.kaiming_uniform_(layer.weight, nonlinearity="relu")
        if layer.bias is not None:
            nn.init.zeros_(layer.bias)

def get_loss_fn(class_weights=None):
    if IS_BINARY:
        if class_weights is not None:
            counts = np.bincount(y_train, minlength=2)
            pos_weight_value = counts[0] / max(counts[1], 1)
            pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32, device=DEVICE)
            return nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        return nn.BCEWithLogitsLoss()
    if class_weights is not None:
        weights = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
        return nn.CrossEntropyLoss(weight=weights)
    return nn.CrossEntropyLoss()

def predict_torch(model: nn.Module, X_values: np.ndarray, threshold: float = 0.5):
    model.eval()
    with torch.no_grad():
        inputs = torch.tensor(X_values, dtype=torch.float32, device=DEVICE)
        logits = model(inputs)
        if IS_BINARY:
            probs = torch.sigmoid(logits).detach().cpu().numpy().reshape(-1)
            return (probs >= threshold).astype(int), probs
        probs = torch.softmax(logits, dim=1).detach().cpu().numpy()
        return probs.argmax(axis=1), probs

def tune_binary_threshold(y_true, probabilities):
    if not IS_BINARY:
        return 0.5
    thresholds = np.linspace(0.1, 0.9, 17)
    scores = []
    for threshold in thresholds:
        pred = (probabilities >= threshold).astype(int)
        scores.append(f1_score(y_true, pred, average="binary", zero_division=0))
    return float(thresholds[int(np.argmax(scores))])

def train_torch_model(
    model_builder,
    model_name: str,
    training_seed: int,
    weight_init_name: str,
    weight_init_tuple,
    epochs: int = 5,
    batch_size: int = 512,
    lr: float = 1e-3,
    use_class_weights: bool = False,
    threshold_tuning: bool = False,
    early_stopping: bool = False,
    scheduler_enabled: bool = False,
):
    set_global_seed(training_seed)
    model = model_builder().to(DEVICE)
    apply_weight_initialization(model, weight_init_tuple)

    class_weights = None
    if use_class_weights and not IS_BINARY:
        present_classes = np.unique(y_train)
        class_weights = np.ones(NUM_CLASSES, dtype=np.float32)
        computed_weights = compute_class_weight(class_weight="balanced", classes=present_classes, y=y_train)
        class_weights[present_classes] = computed_weights
    elif use_class_weights and IS_BINARY:
        class_weights = "binary"

    loss_fn = get_loss_fn(class_weights)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=2, factor=0.5) if scheduler_enabled else None
    loader = get_dataloaders(X_train, y_train, batch_size=batch_size, seed=training_seed)

    best_state = None
    best_loss = float("inf")
    patience = 3
    stale_epochs = 0

    for epoch in range(epochs):
        model.train()
        epoch_losses = []
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            optimizer.step()
            epoch_losses.append(float(loss.item()))

        epoch_loss = float(np.mean(epoch_losses))
        if scheduler is not None:
            scheduler.step(epoch_loss)

        if early_stopping:
            if epoch_loss < best_loss - 1e-5:
                best_loss = epoch_loss
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                stale_epochs = 0
            else:
                stale_epochs += 1
            if stale_epochs >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    threshold = 0.5
    if threshold_tuning and IS_BINARY:
        _, train_prob = predict_torch(model, X_train)
        threshold = tune_binary_threshold(y_train, train_prob)

    y_pred, _ = predict_torch(model, X_test, threshold=threshold)
    metrics = evaluate_predictions(y_test, y_pred, labels=list(label_encoder.classes_))
    return {
        "dataset_name": DATASET_NAME,
        "framework": FRAMEWORK,
        "model_name": model_name,
        "training_seed": training_seed,
        "weight_init_name": weight_init_name,
        "weight_init_tuple": json.dumps(weight_init_tuple),
        **RUN_METADATA,
        "accuracy": metrics["accuracy"],
        "precision": metrics["precision"],
        "recall": metrics["recall"],
        "f1": metrics["f1"],
        "confusion_matrix": metrics["confusion_matrix"],
        "classification_report": metrics["classification_report"],
    }


import gc

def append_result_row(row: dict, csv_path: Path) -> None:
    row_to_save = row.copy()
    for col in ["confusion_matrix", "classification_report"]:
        if col in row_to_save:
            row_to_save[col] = json.dumps(row_to_save[col])
    pd.DataFrame([row_to_save]).to_csv(
        csv_path,
        mode="a",
        index=False,
        header=not csv_path.exists(),
    )

def cleanup_after_run() -> None:
    gc.collect()
    torch.cuda.empty_cache()

def run_model_experiment(model_builder, model_name: str, **train_kwargs) -> pd.DataFrame:
    rows = []
    total_runs = len(TRAINING_SEEDS) * len(WEIGHT_INIT_SEEDS)
    run_index = 0
    for training_seed in TRAINING_SEEDS:
        for weight_init_name, weight_init_tuple in WEIGHT_INIT_SEEDS.items():
            run_index += 1
            print(f"[{model_name}] run {run_index}/{total_runs}: seed={training_seed}, init={weight_init_name}")
            row = train_torch_model(
                model_builder=model_builder,
                model_name=model_name,
                training_seed=training_seed,
                weight_init_name=weight_init_name,
                weight_init_tuple=weight_init_tuple,
                **train_kwargs,
            )
            rows.append(row)
            append_result_row(row, RESULTS_CSV_PATH)
            cleanup_after_run()
    return pd.DataFrame(rows)

### Model definitions used by sections 11-14

The framework-style model below is a CADE-inspired implementation focused on concept-drift and novel-class-aware IDS evaluation; it is not an exact official reproduction unless the original code is provided. The improved MLP is tuned specifically for CSE-CIC-IDS-2018 with class imbalance handling, dropout, batch normalization, early stopping, threshold tuning where applicable, and learning-rate scheduling.

In [ ]:
class FrameworkStyleMLP(nn.Module):
    def __init__(self):
        super().__init__()
        output_dim = 1 if IS_BINARY else NUM_CLASSES
        self.net = nn.Sequential(
            nn.Linear(NUM_FEATURES, 320),
            nn.BatchNorm1d(320),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(320, 160),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(160, output_dim),
        )

    def forward(self, x):
        return self.net(x)


## Train and evaluate CADE-style MLP\n\nThis split notebook executes only this model with the configured seed and initialization grid.\n

In [ ]:
if RESULTS_CSV_PATH.exists():
    RESULTS_CSV_PATH.unlink()

model_results_frames = []
for held_out_attack in HELD_OUT_ATTACKS:
    print(f"\n=== Held-out attack: {held_out_attack} ===")
    prepare_cse_held_out_experiment(held_out_attack)
    result_frame = run_model_experiment(FrameworkStyleMLP, FRAMEWORK_STYLE_MODEL_NAME, epochs=5, batch_size=512, lr=1e-3)
    print("Per-attack aggregate results:")
    display(aggregate_results(result_frame))
    model_results_frames.append(result_frame)

per_run_results = pd.concat(model_results_frames, ignore_index=True)
per_run_results.head()


## Aggregate and save results\n\nThe per-run CSV is appended after every seed and initialization run. The aggregated CSV is saved at the end.\n

In [ ]:
metric_cols = ["accuracy", "precision", "recall", "f1"]
required_detail_cols = ["dataset_name", "framework", "model_name", "seed", "weight_init", *metric_cols]
context_cols = [
    col for col in ["held_out_attack"]
    if col in per_run_results.columns and per_run_results[col].notna().any()
]

detailed_results = per_run_results.copy()
detailed_results["seed"] = detailed_results["training_seed"] if "training_seed" in detailed_results.columns else "default"
detailed_results["weight_init"] = detailed_results["weight_init_name"] if "weight_init_name" in detailed_results.columns else "default"
detailed_results = detailed_results[required_detail_cols + context_cols]

aggregation_group_cols = ["dataset_name", "framework", "model_name"] + context_cols
aggregated_results = detailed_results.groupby(aggregation_group_cols, dropna=False)[metric_cols].agg(
    ["mean", "min", "max", "std"]
).round(6)
aggregated_results.columns = [f"{metric}_{stat}" for metric, stat in aggregated_results.columns]
aggregated_results = aggregated_results.reset_index()

DETAILED_RESULTS_PATH = RESULTS_DIR / "detailed_results.csv"
AGGREGATED_RESULTS_PATH = RESULTS_DIR / "aggregated_results.csv"
detailed_results.to_csv(DETAILED_RESULTS_PATH, index=False)
aggregated_results.to_csv(AGGREGATED_RESULTS_PATH, index=False)

print("=== DETAILED RESULTS (PER SEED) ===")
display(detailed_results)

print("=== AGGREGATED RESULTS ===")
display(aggregated_results)

print("Saved detailed results to:", DETAILED_RESULTS_PATH)
print("Saved aggregated results to:", AGGREGATED_RESULTS_PATH)


## Final comparison table\n

In [ ]:
print("=== DONE ===")
